In [2]:
# Load Packages and Connect to SQL Server
!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://localhost\\SQLEXPRESS/TMDB?"
    "driver=ODBC+Driver+17+for+SQL+Server&"
    "trusted_connection=yes&"
    "TrustServerCertificate=yes"
)

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Users\Bruv\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [18]:
# Helper to run a .sql file
def run_query(filepath):
    with open(filepath, "r") as f:
        sql = f.read()
    print(repr(sql))
    return pd.read_sql(sql, engine)

In [26]:
# Q1: Top Genres by Average Revenue
import matplotlib.ticker as mticker

df1 = run_query("../sql/queries/top_genres_by_revenue.sql")
df1.head(10).plot(kind="bar", x="genre", y="avg_revenue", legend=False,
                   title="Top Genres by Avg Revenue")
plt.ylabel("Average Revenue (in Millions $)")
plt.tight_layout()
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}'))
plt.savefig("../visuals/top_genres_revenue.png")
plt.close()

'-- Top 10 genres by average revenue\nSELECT TOP 10\n    g.name AS genre,\n    COUNT(DISTINCT g.movie_id) AS movie_count,\n    AVG(m.revenue) AS avg_revenue\nFROM genres AS g\nJOIN dbo.movies AS m ON g.movie_id = m.id\nWHERE m.revenue > 0\nGROUP BY g.name\nORDER BY avg_revenue DESC'


In [33]:
# Q2: Top Directors by Average Rating (min. 3 films)
df2 = run_query("../sql/queries/top_directors_by_rating.sql")
df2.head(10).plot(kind="bar", x="director", y="avg_rating", legend=False,
                   title="Top Directors by Avg Rating")
plt.ylabel("Average Rating")
plt.ylim(7.4,8.1)
plt.tight_layout()
plt.savefig("../visuals/top_directors.png")
plt.close()

"-- Directors with the highest average rating (minimum 3 films)\nSELECT TOP 50\n    c.name AS director,\n    COUNT(DISTINCT c.movie_id) AS film_count,\n    AVG(m.vote_average) AS avg_rating\nFROM crew_members c\nJOIN dbo.movies m ON c.movie_id = m.id\nWHERE c.job = 'Director'\nGROUP BY c.name\nHAVING COUNT(DISTINCT c.movie_id) >= 3\nORDER BY avg_rating DESC"


In [ ]:
# Q3: Highest Actors' Collaborations
df3 = run_query("../sql/queries/actor_collaborations.sql")
df3.columns = ["Actor 1", "Actor 2", "Movies Together"]
df3["Pair"] = df3["Actor 1"] + " & " + df3["Actor 2"]
df3.head(10).plot(kind="barh", x="Pair", y="Movies Together", legend=False,
                  title="Most Actor Collaborations")
plt.xlabel("Number of Movies Together")
plt.xlim(8,13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../visuals/most_actor_collaborations.png")
plt.close()

'-- Top 10 actor pairs who have appeared together most frequently\nSELECT TOP 10\n    a.actor_name AS actor_1,\n    b.actor_name AS actor_2,\n    COUNT(*) AS movies_together\nFROM cast_members AS a\nJOIN cast_members AS b\n    ON a.movie_id = b.movie_id\n    AND a.actor_name < b.actor_name\nGROUP BY a.actor_name, b.actor_name\nHAVING COUNT(*) > 1\nORDER BY movies_together DESC'


In [53]:
# Q4: Most Profitable Movies
df4 = run_query("../sql/queries/most_profitable_movies.sql")
df4.head(10).plot(kind="barh", x="title", y="profit", legend=False,
                   title="Most Profitable Movies")
plt.xlabel("Profit (in Billions $)")
plt.yticks(fontsize=8)
plt.xlim(0.5e9,3e9)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../visuals/top_profitable_movies.png")
plt.close()

'-- Most profitable movies (revenue - budget), excluding missing data\nSELECT TOP 50\n    title,\n    budget,\n    revenue,\n    (revenue - budget) AS profit\nFROM dbo.movies\nWHERE budget > 0 AND revenue > 0\nORDER BY profit DESC'


In [55]:
# Q5: Budget & Revenue Trend by Year
df5 = run_query("../sql/queries/budget_revenue_trend_by_year.sql")
df5.plot(x="release_year", y=["avg_budget", "avg_revenue"],
          title="Budget vs Revenue Over Time")
plt.ylabel("Amount ($)")
plt.tight_layout()
plt.savefig("../visuals/budget_revenue_trend.png")
plt.close()

'-- Average budget and revenue trend by release year\nSELECT\n   YEAR(release_date) AS release_year,\n   AVG(budget) AS avg_budget,\n   AVG(revenue) AS avg_revenue\nFROM dbo.movies\nWHERE release_date IS NOT NULL\n   AND budget > 1000 AND revenue > 1000\nGROUP BY YEAR(release_date)\nORDER BY release_year'
